# Demo: Módulos do Pipeline de Transcrição

Este notebook demonstra cada módulo do pipeline de forma isolada:
1. `transcription.py` — ASR com Whisper (Transformers)
2. `diarization.py` — Identificação de speakers com pyannote.audio
3. `alignment.py` — Merge de transcrição + diarização
4. `export.py` — Geração de `.md` e `.pdf`

In [1]:
import sys
from pathlib import Path

# Adiciona o projeto ao path
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print("Project root:", project_root)

Project root: d:\UC15\senac_ia_uc15_nlp_project\notebooks\pablo\meeting_transcription


In [2]:
from dotenv import load_dotenv
load_dotenv(project_root / ".env")

from src.config import build_settings
cfg = build_settings()
print("Whisper model:", cfg.whisper_model)
print("Language:", cfg.language)
print("HF_TOKEN configurado:", bool(cfg.hf_token))

Whisper model: large-v3
Language: pt
HF_TOKEN configurado: True


## 1. Transcrição com Whisper

Coloque um arquivo de áudio em `data/audio/` e ajuste o caminho abaixo.

In [3]:
AUDIO_FILE = cfg.audio_dir / "audio_01.ogg"  # <- Ajuste o nome do arquivo

print("Arquivo de áudio:", AUDIO_FILE)
print("Existe:", AUDIO_FILE.exists())

Arquivo de áudio: D:\UC15\senac_ia_uc15_nlp_project\notebooks\pablo\meeting_transcription\data\audio\audio_01.ogg
Existe: True


In [4]:
from src.transcription import transcribe_audio

transcript_result = transcribe_audio(
    audio_path=AUDIO_FILE,
    model_name=cfg.whisper_model,
    language=cfg.language,
)

print(f"Idioma detectado: {transcript_result.get('language')}")
print(f"Total de segmentos: {len(transcript_result.get('segments', []))}")
print("\nPrimeiros 3 segmentos:")
for seg in transcript_result["segments"][:3]:
    print(f"  [{seg['start']:.1f}s → {seg['end']:.1f}s] {seg['text']}")

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/1259 [00:00<?, ?it/s]

A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> to see related `.generate()` flags.
You have passed `return_dict_in

Idioma detectado: pt
Total de segmentos: 6

Primeiros 3 segmentos:
  [0.0s → 5.5s] Nossa, me deu um trem ruim agora à noite, lá na casa da mulher lá.
  [5.5s → 10.1s] Eu não sei se eu tava suando frio ou se eu tava suando calor.
  [10.1s → 15.1s] Nossa, é um trem mais esquisito, uma sensação assim meio que de desmaio.


## 2. Diarização com pyannote.audio

> Necessário: `HF_TOKEN` configurado e modelos aceitos no Hugging Face Hub.

In [5]:
from src.diarization import diarize_audio

diarization = diarize_audio(
    audio_path=AUDIO_FILE,
    hf_token=cfg.hf_token,
    num_speakers=cfg.num_speakers,
    min_speakers=cfg.min_speakers,
    max_speakers=cfg.max_speakers,
)

print("Tipo do resultado:", type(diarization))
print("\nPrimeiros 5 turns:")
for i, (turn, _, speaker) in enumerate(diarization.itertracks(yield_label=True)):
    if i >= 5:
        break
    print(f"  [{turn.start:.1f}s -> {turn.end:.1f}s] {speaker}")

LocalEntryNotFoundError: An error happened while trying to locate the file on the Hub and we cannot find the requested files in the local cache. Please check your connection and try again or make sure your Internet connection is on.

## 3. Alinhamento: Transcrição + Diarização

In [ ]:
from src.alignment import assign_speakers

segments = assign_speakers(transcript_result, diarization)

print(f"Total de segmentos com speaker: {len(segments)}")
print("\nPrimeiros 5 segmentos:")
for seg in segments[:5]:
    print(f"  [{seg.start:.1f}s → {seg.end:.1f}s] {seg.speaker}: {seg.text}")

## 4. Exportação para `.md` e `.pdf`

In [ ]:
from src.export import to_markdown, to_pdf

md_path = to_markdown(
    segments=segments,
    output_path=cfg.output_dir / "demo_transcricao.md",
    title="Demo — Transcrição de Reunião",
)
pdf_path = to_pdf(
    segments=segments,
    output_path=cfg.output_dir / "demo_transcricao.pdf",
    title="Demo — Transcrição de Reunião",
)

print("Markdown gerado:", md_path)
print("PDF gerado:", pdf_path)

In [ ]:
# Visualizar início do Markdown gerado
print(md_path.read_text(encoding="utf-8")[:1000])